In [10]:
# =========================================================================
# 1. LIBRERÍAS Y CONFIGURACIÓN DE MLFLOW
# =========================================================================
import mlflow
import mlflow.sklearn
import mlflow.tensorflow
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Conectar con el servidor local de MLflow
mlflow.set_tracking_uri("http://localhost:5000")

In [11]:
# =========================================================================
# 2. CARGA DE DATOS (TEST)
# =========================================================================
# Cargar el parquet generado por Airflow
ruta_test = "../../airflow/plugins/09_test.parquet"
df_test = pd.read_parquet(ruta_test)

# Separar variables (X) de la etiqueta (y) para el modelo de Sepsis
X_test_sepsis = df_test.drop(columns=['SepsisLabel'])
y_test_sepsis = df_test['SepsisLabel'].values

print(f"Datos de Sepsis cargados: {X_test_sepsis.shape[0]} pacientes y {X_test_sepsis.shape[1]} variables.")

Datos de Sepsis cargados: 303712 pacientes y 38 variables.


In [14]:
# Clasificador_RF_Sepsis
import mlflow.sklearn
import pandas as pd

mlflow.set_tracking_uri("http://localhost:5000")
modelo_sepsis = mlflow.sklearn.load_model("models:/Clasificador_RF_Sepsis/latest")

df_test = pd.read_parquet("../../airflow/plugins/09_test.parquet")
X_test = df_test.drop(columns=['SepsisLabel'])
y_test = df_test['SepsisLabel'].values

casos_sepsis = {
    "Paciente Real 1": X_test[y_test == 0].iloc[[0]],
    "Paciente Real 2": X_test[y_test == 1].iloc[[0]]
}

UMBRAL_ALERTA = 20.0  

for nombre, datos in casos_sepsis.items():
    pred_proba = modelo_sepsis.predict_proba(datos.to_numpy())[0][1] * 100
    pred_clase = 1 if pred_proba >= UMBRAL_ALERTA else 0
    
    print("\n" + nombre + ":")
    print("Probabilidad Sepsis:", pred_proba)
    if pred_clase == 1:
        print("Diagnostico: ALERTA SEPSIS (1)")
    else:
        print("Diagnostico: ESTABLE (0)")


Paciente Real 1:
Probabilidad Sepsis: 12.244767066193967
Diagnostico: ESTABLE (0)

Paciente Real 2:
Probabilidad Sepsis: 31.722896130461674
Diagnostico: ALERTA SEPSIS (1)


In [15]:
# Clasificador_RF_Triaje
import mlflow.sklearn
import pandas as pd

mlflow.set_tracking_uri("http://localhost:5000")
modelo_rf = mlflow.sklearn.load_model("models:/Clasificador_RF_Triaje/latest")

pacientes_triaje = pd.DataFrame(
    [
        [36.5, 75.0, 14.0, 98.0, 120.0, 80.0, 2.0],
        [39.2, 110.0, 20.0, 95.0, 110.0, 70.0, 6.0],
        [35.1, 140.0, 30.0, 88.0, 80.0, 45.0, 9.0]
    ],
    columns=['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain'],
    index=["Paciente 1", "Paciente 2", "Paciente 3"]
)

for paciente in pacientes_triaje.index:
    datos = pacientes_triaje.loc[[paciente]]
    pred_proba = modelo_rf.predict_proba(datos)[0][1] * 100
    print("\n" + paciente + ":")
    print("Probabilidad Urgencia:", pred_proba)


Paciente 1:
Probabilidad Urgencia: 43.8

Paciente 2:
Probabilidad Urgencia: 56.39999999999999

Paciente 3:
Probabilidad Urgencia: 23.599999999999998


In [16]:
# Clasificador_RF_Triaje_m
import mlflow.sklearn
import pandas as pd

mlflow.set_tracking_uri("http://localhost:5000")
modelo_rf_m = mlflow.sklearn.load_model("models:/Clasificador_RF_Triaje_m/latest")

columnas = modelo_rf_m.feature_names_in_
pacientes_triaje = pd.DataFrame(0.0, index=["Paciente 1", "Paciente 2", "Paciente 3"], columns=columnas)

v1 = [36.5, 75.0, 14.0, 98.0, 120.0, 80.0, 2.0]
v2 = [39.2, 110.0, 20.0, 95.0, 110.0, 70.0, 6.0]
v3 = [35.1, 140.0, 30.0, 88.0, 80.0, 45.0, 9.0]
nombres_v = ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain']

for col in nombres_v:
    if col in columnas:
        pacientes_triaje.loc["Paciente 1", col] = v1[nombres_v.index(col)]
        pacientes_triaje.loc["Paciente 2", col] = v2[nombres_v.index(col)]
        pacientes_triaje.loc["Paciente 3", col] = v3[nombres_v.index(col)]

if 'shock_index' in columnas:
    pacientes_triaje['shock_index'] = pacientes_triaje['heartrate'] / pacientes_triaje['sbp']
if 'pulse_pressure' in columnas:
    pacientes_triaje['pulse_pressure'] = pacientes_triaje['sbp'] - pacientes_triaje['dbp']

for paciente in pacientes_triaje.index:
    datos = pacientes_triaje.loc[[paciente]]
    pred_proba = modelo_rf_m.predict_proba(datos)[0][1] * 100
    print("\n" + paciente + ":")
    print("Probabilidad Urgencia:", pred_proba)


Paciente 1:
Probabilidad Urgencia: 42.81222807407019

Paciente 2:
Probabilidad Urgencia: 49.84958060944902

Paciente 3:
Probabilidad Urgencia: 51.53462843199685


In [17]:
# Clasificador_TF_Triaje
import os
import warnings
import mlflow.tensorflow
import pandas as pd

# Silenciar advertencias de TensorFlow y Keras
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore', category=UserWarning)

mlflow.set_tracking_uri("http://localhost:5000")
modelo_tf = mlflow.tensorflow.load_model("models:/Clasificador_TF_Triaje/latest")

pacientes_triaje = pd.DataFrame(
    [
        [36.5, 75.0, 14.0, 98.0, 120.0, 80.0, 2.0],
        [39.2, 110.0, 20.0, 95.0, 110.0, 70.0, 6.0],
        [35.1, 140.0, 30.0, 88.0, 80.0, 45.0, 9.0]
    ],
    columns=['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain'],
    index=["Paciente 1", "Paciente 2", "Paciente 3"]
)

predicciones = modelo_tf.predict(pacientes_triaje, verbose=0)

for i, paciente in enumerate(pacientes_triaje.index):
    probabilidad = predicciones[i][0] * 100
    print("\n" + paciente + ":")
    print("Probabilidad Urgencia:", probabilidad)


Paciente 1:
Probabilidad Urgencia: 91.97882

Paciente 2:
Probabilidad Urgencia: 92.036995

Paciente 3:
Probabilidad Urgencia: 89.2477
